# TIFF Folder to Stack

Combine a folder of individual TIFF slices (e.g. XCT reconstruction output) into a single multi-page (Big)TIFF stack.

**Requirements:** `pip install tifffile numpy natsort`

Notes:
- Uses BigTIFF automatically so 2000+ slice CT stacks aren't limited by the classic 4GB TIFF cap.
- Writes slice-by-slice instead of loading the whole volume into RAM first.
- Slices are sorted "naturally" (`slice_2.tif` before `slice_10.tif`) rather than alphabetically.
- Validates that every slice matches the first slice's shape/dtype before writing.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import tifffile
from natsort import natsorted

## 1. Set your paths

Edit these two variables, then run the rest of the notebook top to bottom.

In [ ]:
# --- EDIT THESE ---
INPUT_FOLDER = Path("/path/to/scan1_slices")
OUTPUT_FILE = Path("/path/to/scan1_stack.tif")
DRY_RUN = True  # set to False once you've checked the ordering below
# -------------------

## 2. Helper functions

In [ ]:
def find_tiffs(folder: Path):
    exts = (".tif", ".tiff")
    files = [f for f in folder.iterdir() if f.suffix.lower() in exts]
    if not files:
        raise FileNotFoundError(f"No .tif/.tiff files found in {folder}")
    return natsorted(files, key=lambda p: p.name)


def build_stack(folder: Path, output_path: Path, dry_run: bool = False):
    files = find_tiffs(folder)
    print(f"Found {len(files)} TIFF files in {folder}")

    # Inspect first file to get shape/dtype and confirm the rest match
    first = tifffile.imread(files[0])
    shape = first.shape
    dtype = first.dtype
    print(f"Reference slice: {files[0].name}  shape={shape}  dtype={dtype}")

    if dry_run:
        print("Dry run only - listing first/last 5 files in stitch order:")
        for f in files[:5]:
            print(f"  {f.name}")
        print("  ...")
        for f in files[-5:]:
            print(f"  {f.name}")
        return

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with tifffile.TiffWriter(output_path, bigtiff=True) as writer:
        for i, f in enumerate(files):
            img = tifffile.imread(f)

            if img.shape != shape:
                raise ValueError(
                    f"Shape mismatch at slice {i} ({f.name}): "
                    f"expected {shape}, got {img.shape}"
                )
            if img.dtype != dtype:
                raise ValueError(
                    f"Dtype mismatch at slice {i} ({f.name}): "
                    f"expected {dtype}, got {img.dtype}"
                )

            writer.write(img, contiguous=True)

            if (i + 1) % 200 == 0 or (i + 1) == len(files):
                print(f"  wrote {i + 1}/{len(files)} slices")

    print(f"Done. Stack written to {output_path}")
    print(f"Final volume shape: ({len(files)}, {shape[0]}, {shape[1]})  dtype={dtype}")

## 3. Dry run

Check that the slice ordering looks right *before* writing a large stack. Confirm the naming sorts correctly (e.g. `slice_0001.tif` vs `slice_1.tif` can sort very differently depending on padding).

In [ ]:
build_stack(INPUT_FOLDER, OUTPUT_FILE, dry_run=True)

## 4. Build the stack

Once the ordering above looks correct, run this cell to actually write the stack.

In [ ]:
build_stack(INPUT_FOLDER, OUTPUT_FILE, dry_run=False)

## 5. (Optional) Quick sanity check

Load the resulting stack and view the middle slice to confirm it wrote correctly.

In [ ]:
import matplotlib.pyplot as plt

stack = tifffile.imread(OUTPUT_FILE)
print("Stack shape:", stack.shape, "dtype:", stack.dtype)

mid = stack.shape[0] // 2
plt.figure(figsize=(6, 6))
plt.imshow(stack[mid], cmap="gray")
plt.title(f"Middle slice ({mid})")
plt.axis("off")
plt.show()

## 6. Repeat for each scan

Run this notebook once per scan (e.g. three times for three scans), changing `INPUT_FOLDER` and `OUTPUT_FILE` in step 1 each time.